# 07 — Robustness: model còn lại bao nhiêu khi gặp mưa, sương, tối?

**Chủ: Phong Nguyễn** · code: `src/gtsrb/robustness/`

## ★ LUẬT SẮT: nhiễu CHỈ áp ở TEST, KHÔNG BAO GIỜ ở TRAIN

Robustness nghĩa là khái quát sang **phân phối chưa từng thấy** (distribution shift).
Train trên các loại nhiễu này rồi test trên chúng là test **in-distribution** — chỉ
chứng minh "model học được cái nó đã thấy". Đó là augmentation, không phải robustness.

`tests/test_corruptions.py` khoá luật này: nó khẳng định `data/transforms.py`
(augmentation lúc train) **không** import `robustness/corruptions.py`.

Chạy `make robustness` trước.

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

## 1. Năm loại nhiễu và mô hình vật lý

In [ ]:
from gtsrb.robustness.corruptions import CORRUPTIONS, SEVERITIES, apply_corruption
from gtsrb.data.preprocess import load_and_process

index = pd.read_csv("data/processed/index.csv")
row = index[index["split"] == "test"].iloc[7]
base = load_and_process(row.path, (row.roi_x1, row.roi_y1, row.roi_x2, row.roi_y2),
                        96, "clahe")
rng = np.random.default_rng(0)

fig, axes = plt.subplots(len(CORRUPTIONS), len(SEVERITIES) + 1,
                         figsize=(2.0 * (len(SEVERITIES) + 1), 2.0 * len(CORRUPTIONS)))
ten_vi = {"motion_blur": "Mờ chuyển động", "gauss_noise": "Nhiễu Gaussian",
          "fog": "Sương mù", "low_light": "Thiếu sáng", "occlusion": "Bị che"}
for r, kind in enumerate(CORRUPTIONS):
    axes[r][0].imshow(base); axes[r][0].set_ylabel(ten_vi[kind], fontsize=9)
    if r == 0: axes[r][0].set_title("gốc", fontsize=9)
    for c, sev in enumerate(SEVERITIES, start=1):
        axes[r][c].imshow(apply_corruption(base, kind, sev, rng))
        if r == 0: axes[r][c].set_title(f"mức {sev}", fontsize=9)
for rr in axes:
    for ax in rr: ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"5 loại nhiễu x 5 mức — {CLASS_NAMES[row.class_id]}", fontsize=12)
plt.tight_layout(); plt.savefig("reports/figures/corruption_grid.png", dpi=130); plt.show()

**Mô hình vật lý của từng loại** (không phải bịa ra cho đẹp):

| Nhiễu | Mô hình |
|---|---|
| Mờ chuyển động | tích chập với kernel **đường thẳng**, dài 3→15 px, góc ngẫu nhiên |
| Nhiễu Gaussian | `I' = I + N(0, σ²)`, σ = 5/10/20/40/60 trên thang 255 |
| Sương mù | **tán xạ khí quyển (Koschmieder)**: `I = J·t + A·(1−t)`, `t = e^(−βd)` |
| Thiếu sáng | gamma `I' = 255·(I/255)^γ`, γ = 1,5→3,2, kèm **nhiễu Poisson** (nhiễu photon thật của ảnh tối, không phải Gaussian) |
| Bị che | hình chữ nhật che 10/20/30/40/50% diện tích |

## 2. Đường cong accuracy theo mức nhiễu

In [ ]:
path = pathlib.Path("reports/tables/robustness.csv")
if not path.exists():
    print("Chưa có bảng. Chạy:", "make robustness")
    rob = pd.DataFrame()      # để các ô sau tự bỏ qua, không vỡ notebook
else:
    rob = pd.read_csv(path)
    print(f"{len(rob)} dòng — {rob['model'].nunique()} model x "
          f"({len(CORRUPTIONS)} nhiễu x {len(SEVERITIES)} mức + 1 clean)")
    rob.head()


In [ ]:
if rob.empty:
    print("chưa có số liệu — chạy make robustness")
else:
    fig, axes = plt.subplots(1, len(CORRUPTIONS), figsize=(4 * len(CORRUPTIONS), 4), sharey=True)
    for ax, kind in zip(axes, CORRUPTIONS):
        part = rob[rob["corruption"] == kind]
        for m, g in part.groupby("model"):
            g = g.sort_values("severity")
            ax.plot(g["severity"], g["top1"] * 100, marker="o", label=m)
        for m, g in rob[rob["corruption"] == "clean"].groupby("model"):
            ax.axhline(float(g["top1"].iloc[0]) * 100, ls=":", alpha=.4)
        ax.set_title(ten_vi[kind]); ax.set_xlabel("mức độ")
        ax.set_xticks(list(SEVERITIES)); ax.grid(alpha=.3)
    axes[0].set_ylabel("test top-1 (%)"); axes[-1].legend(fontsize=8)
    fig.suptitle("Đường chấm = accuracy trên ảnh sạch", fontsize=11)
    plt.tight_layout(); plt.show()

## 3. ★ Relative robustness — tách "giỏi sẵn" khỏi "BỀN"

```
relative = acc_nhiễu / acc_sạch
```

Nếu chỉ báo accuracy **tuyệt đối** dưới nhiễu thì model nào giỏi sẵn sẽ luôn thắng,
và ta **không học được gì** về tính bền. Chia cho accuracy sạch tách hai thứ đó ra.

In [ ]:
summary_path = pathlib.Path("reports/tables/robustness_summary.csv")
if summary_path.exists():
    summ = pd.read_csv(summary_path, index_col=0)
    print(summ.round(4).to_string())
    print()
    if len(summ) >= 2:
        best_clean = summ["clean_top1"].idxmax()
        best_rob = summ.drop(columns=["mCE", "clean_top1"]).mean(axis=1).idxmax()
        print(f"Accuracy SẠCH cao nhất : {best_clean}")
        print(f"BỀN nhất (relative)    : {best_rob}")
        if best_clean != best_rob:
            print()
            print(">>> HAI MODEL KHÁC NHAU! Đây là phát hiện đáng giá nhất của phần này:")
            print(">>> model accuracy cao nhất KHÔNG phải model bền nhất.")
        else:
            print("\n(Cùng một model — vẫn phải báo cáo relative để chứng minh đã kiểm.)")
else:
    print("Chưa có robustness_summary.csv. Chạy: make robustness")

In [ ]:
# Heatmap relative robustness
if summary_path.exists():
    summ = pd.read_csv(summary_path, index_col=0)
    cols = [c for c in summ.columns if c in CORRUPTIONS]
    if cols and len(summ):
        fig, ax = plt.subplots(figsize=(1.6 * len(cols) + 3, 0.9 * len(summ) + 2))
        im = ax.imshow(summ[cols].values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels([ten_vi.get(c, c) for c in cols], rotation=25, ha="right")
        ax.set_yticks(range(len(summ))); ax.set_yticklabels(summ.index)
        for i in range(len(summ)):
            for j, c in enumerate(cols):
                ax.text(j, i, f"{summ[c].iloc[i]:.2f}", ha="center", va="center", fontsize=9)
        ax.set_title("Relative robustness (acc_nhiễu / acc_sạch) — 1,0 là không mất gì")
        fig.colorbar(im, ax=ax)
        plt.tight_layout(); plt.show()

**Giả thuyết của nhóm** (có thể bị bác bỏ — bị bác bỏ cũng là kết quả tốt):

- Model pretrained ImageNet **bền hơn** trước nhiễu photometric (noise, low-light, fog),
  vì 1,28 triệu ảnh ImageNet vốn đã có đủ loại điều kiện chụp.
- Model dùng **GAP + spatial dropout** bền hơn trước **occlusion**, vì nó không dựa vào
  một vùng không gian cố định nào.
- **Model accuracy sạch cao nhất không chắc là model bền nhất.**

**mCE** (kiểu ImageNet-C) = trung bình error qua mọi nhiễu và mọi mức → một con số
duy nhất để xếp hạng. Càng **thấp** càng tốt.

## 4. Kết luận cho báo cáo

Phát biểu kết luận theo mẫu này, với số thật thay vào:

> Trên ảnh sạch, **M?** đạt top-1 cao nhất (**?**%). Nhưng dưới sương mù mức 4,
> accuracy còn **?**%, và dưới occlusion 30% còn **?**%. Xét **relative robustness**,
> model bền nhất là **M?** (giữ được **?**% hiệu năng trung bình).
> Vì điều kiện vận hành thật trên đường có mưa, sương và đèn ngược, nhóm khuyến nghị
> **M?** cho triển khai — xem thêm Pareto accuracy–latency ở `notebooks/09`.

**Câu bẫy giảng viên hay hỏi:** *"Accuracy 99% rồi, dùng được trên xe thật chưa?"*
→ **Chưa.** (1) phân phối test giống train (cùng chụp ở Đức, cùng camera);
(2) robustness tụt rất nhanh — chính bảng này là bằng chứng;
(3) đây là bài **phân loại**, không phải **phát hiện** — model nhận biển báo
**đã cắt sẵn**, hệ thống thật phải tự tìm biển báo trong khung ảnh toàn cảnh trước,
và lỗi bước đó cộng dồn vào.